**#1.** You may use the provided dataset with 377 engineered features or create your own feature set. Starting from a large number of features, you should reduce the set to 20 or fewer features. Use the tools we discussed to help you in this work.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestClassifier
from feature_engine.encoding import RareLabelEncoder, CountFrequencyEncoder
from feature_engine.discretisation import EqualFrequencyDiscretiser
from feature_engine.selection import DropConstantFeatures

df = pd.read_csv('adult.csv')

df['income'] = df['income'].astype(str).str.strip().replace({'>50K.': '>50K', '<=50K.': '<=50K'})
df['income'] = (df['income'] == '>50K').astype(int)
df['gender'] = (df['gender'].astype(str).str.strip() == 'Male').astype(int)
df = df.replace('?', np.nan)
df = df.drop(columns=['fnlwgt'])

for c in df.columns:
    if df[c].dtype == 'object':
        df[c] = df[c].fillna('unknown')
    else:
        df[c] = df[c].fillna(df[c].median())

X = df.drop(columns=['income'])
y = df['income']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

rare = RareLabelEncoder(tol=0.01, variables=cat_cols)
X_train = rare.fit_transform(X_train)
X_test = rare.transform(X_test)

freq = CountFrequencyEncoder(variables=cat_cols, encoding_method='frequency')
X_train = freq.fit_transform(X_train)
X_test = freq.transform(X_test)

disc_vars = [c for c in num_cols if c not in ['gender', 'capital-gain', 'capital-loss']]
disc = EqualFrequencyDiscretiser(q=5, variables=disc_vars)
X_train = disc.fit_transform(X_train)
X_test = disc.transform(X_test)

drop_const = DropConstantFeatures()
X_train = drop_const.fit_transform(X_train)
X_test = drop_const.transform(X_test)

poly = PolynomialFeatures(degree=3, interaction_only=True, include_bias=False)
poly_names = poly.fit(X_train).get_feature_names_out(X_train.columns)
X_train_377 = pd.DataFrame(poly.transform(X_train), columns=poly_names, index=X_train.index)
X_test_377 = pd.DataFrame(poly.transform(X_test), columns=poly_names, index=X_test.index)

print('Engineered feature count:', X_train_377.shape[1])

rf = RandomForestClassifier(random_state=42, class_weight='balanced')
rf.fit(X_train_377, y_train)
imp = pd.Series(rf.feature_importances_, index=X_train_377.columns).sort_values(ascending=False)
top20 = imp.head(20).index.tolist()

X_train_reduced = X_train_377[top20].copy()
X_test_reduced = X_test_377[top20].copy()

print('Reduced feature count:', X_train_reduced.shape[1])
print('Top 20 features:')
for f in top20:
    print('-', f)


/opt/anaconda3/lib/python3.13/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable workclass is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable marital-status is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable relationship is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable race is less than that indicated in n_categories.

Engineered feature count: 377
Reduced feature count: 20
Top 20 features:
- marital-status
- marital-status native-country
- age marital-status occupation
- age marital-status hours-per-week
- age educational-num marital-status
- marital-status hours-per-week
- marital-status occupation native-country
- age marital-status
- marital-status race
- age marital-status race
- age relationship hours-per-week
- marital-status race native-country
- age educational-num hours-per-week
- marital-status occupation race
- age marital-status relationship
- marital-status occupation hours-per-week
- marital-status occupation
- educational-num marital-status occupation
- marital-status relationship
- educational-num marital-status relationship


**#2.** Train at least two models that differ in a meaningful way, such as model type or tuning choices, and use your reduced feature sets for these models.

In [2]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score

needed = ['X_train_reduced', 'X_test_reduced', 'y_train', 'y_test']
for n in needed:
    if n not in globals():
        raise ValueError(f'Missing {n}. Run Step 1 first.')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_base = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_base.fit(X_train_reduced, y_train)
log_base_score = balanced_accuracy_score(y_test, log_base.predict(X_test_reduced))

log_grid = {'C': [0.1, 1, 10], 'solver': ['liblinear', 'lbfgs']}
log_search = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    log_grid, scoring='balanced_accuracy', cv=cv, n_jobs=-1
)
log_search.fit(X_train_reduced, y_train)
log_tuned = log_search.best_estimator_
log_tuned_score = balanced_accuracy_score(y_test, log_tuned.predict(X_test_reduced))

rf_base = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_base.fit(X_train_reduced, y_train)
rf_base_score = balanced_accuracy_score(y_test, rf_base.predict(X_test_reduced))

rf_grid = {'n_estimators': [200, 400], 'max_depth': [None, 10], 'min_samples_leaf': [1, 3]}
rf_search = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    rf_grid, scoring='balanced_accuracy', cv=cv, n_jobs=-1
)
rf_search.fit(X_train_reduced, y_train)
rf_tuned = rf_search.best_estimator_
rf_tuned_score = balanced_accuracy_score(y_test, rf_tuned.predict(X_test_reduced))

print('Logistic Regression - baseline:', round(log_base_score, 4))
print('Logistic Regression - tuned   :', round(log_tuned_score, 4))
print('Best Logistic params:', log_search.best_params_)
print('Random Forest - baseline:', round(rf_base_score, 4))
print('Random Forest - tuned   :', round(rf_tuned_score, 4))
print('Best RF params:', rf_search.best_params_)


Logistic Regression - baseline: 0.7805
Logistic Regression - tuned   : 0.7805
Best Logistic params: {'C': 1, 'solver': 'liblinear'}
Random Forest - baseline: 0.7759
Random Forest - tuned   : 0.7998
Best RF params: {'max_depth': 10, 'min_samples_leaf': 3, 'n_estimators': 400}


**#3.** Tune your models beyond their default settings. There is no fixed number of parameters you must tune, but you should make a reasonable effort to improve performance and demonstrate that your tuning choices had an effect.

In [4]:
import time
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score

needed = ['X_train_reduced', 'X_test_reduced', 'y_train', 'y_test', 'log_tuned_score', 'rf_tuned_score']
for n in needed:
    if n not in globals():
        raise ValueError(f"Missing {n}. Run Step 1 and Step 2 first.")

start = time.time()
cv_fast = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

log_params = {
    "C": [0.05, 0.1, 0.5, 1, 2, 5, 10],
    "solver": ["liblinear"],
    "penalty": ["l1", "l2"]
}

log_search_step3 = RandomizedSearchCV(
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    param_distributions=log_params,
    n_iter=6,
    scoring="balanced_accuracy",
    cv=cv_fast,
    n_jobs=-1,
    random_state=42
)
log_search_step3.fit(X_train_reduced, y_train)
log_tuned_step3 = log_search_step3.best_estimator_
log_tuned_step3_score = balanced_accuracy_score(y_test, log_tuned_step3.predict(X_test_reduced))

rf_params = {
    "n_estimators": [100, 150, 200, 250],
    "max_depth": [6, 8, 10, None],
    "min_samples_leaf": [1, 2, 3, 5],
    "min_samples_split": [2, 5, 10],
    "max_features": ["sqrt", "log2", None]
}

rf_search_step3 = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced", n_jobs=1),
    param_distributions=rf_params,
    n_iter=8,   
    scoring="balanced_accuracy",
    cv=cv_fast,
    n_jobs=-1,
    random_state=42
)
rf_search_step3.fit(X_train_reduced, y_train)
rf_tuned_step3 = rf_search_step3.best_estimator_
rf_tuned_step3_score = balanced_accuracy_score(y_test, rf_tuned_step3.predict(X_test_reduced))

print("Logistic - Step 2 tuned:", round(log_tuned_score, 4))
print("Logistic - Step 3 tuned:", round(log_tuned_step3_score, 4))
print("Best Logistic Step 3 params:", log_search_step3.best_params_)
print("RF - Step 2 tuned:", round(rf_tuned_score, 4))
print("RF - Step 3 tuned:", round(rf_tuned_step3_score, 4))
print("Best RF Step 3 params:", rf_search_step3.best_params_)

best_model_name = "Logistic Regression (Step 3 tuned)" if log_tuned_step3_score >= rf_tuned_step3_score else "Random Forest (Step 3 tuned)"
best_score = max(log_tuned_step3_score, rf_tuned_step3_score)



/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalt

Logistic - Step 2 tuned: 0.7805
Logistic - Step 3 tuned: 0.7809
Best Logistic Step 3 params: {'solver': 'liblinear', 'penalty': 'l2', 'C': 0.5}
RF - Step 2 tuned: 0.7998
RF - Step 3 tuned: 0.8006
Best RF Step 3 params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': 'log2', 'max_depth': 8}


**#4.** Combine your models using stacking by implementing a meta learner that uses out-of-fold predictions, and you should compare the performance of your base models with the stacked model.

In [5]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score

needed = ['X_train_reduced', 'X_test_reduced', 'y_train', 'y_test']
for n in needed:
    if n not in globals():
        raise ValueError(f"Missing {n}. Run Step 1-3 first.")

if 'log_tuned_step3' in globals() and 'rf_tuned_step3' in globals():
    model_log = log_tuned_step3
    model_rf = rf_tuned_step3
    model_label = "Step 3 tuned"
elif 'log_tuned' in globals() and 'rf_tuned' in globals():
    model_log = log_tuned
    model_rf = rf_tuned
    model_label = "Step 2 tuned"
else:
    raise ValueError("No tuned base models found. Run Step 2 (and Step 3 if using it).")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_oof = np.zeros(len(X_train_reduced))
rf_oof = np.zeros(len(X_train_reduced))

for train_idx, val_idx in cv.split(X_train_reduced, y_train):
    X_tr, X_val = X_train_reduced.iloc[train_idx], X_train_reduced.iloc[val_idx]
    y_tr = y_train.iloc[train_idx]

    log_fold = clone(model_log)
    rf_fold = clone(model_rf)

    log_fold.fit(X_tr, y_tr)
    rf_fold.fit(X_tr, y_tr)

    log_oof[val_idx] = log_fold.predict_proba(X_val)[:, 1]
    rf_oof[val_idx] = rf_fold.predict_proba(X_val)[:, 1]

X_meta_train = pd.DataFrame({
    'log_prob': log_oof,
    'rf_prob': rf_oof
})

meta = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
meta.fit(X_meta_train, y_train)

log_full = clone(model_log)
rf_full = clone(model_rf)

log_full.fit(X_train_reduced, y_train)
rf_full.fit(X_train_reduced, y_train)

X_meta_test = pd.DataFrame({
    'log_prob': log_full.predict_proba(X_test_reduced)[:, 1],
    'rf_prob': rf_full.predict_proba(X_test_reduced)[:, 1]
})

log_score = balanced_accuracy_score(y_test, log_full.predict(X_test_reduced))
rf_score = balanced_accuracy_score(y_test, rf_full.predict(X_test_reduced))
stack_score = balanced_accuracy_score(y_test, meta.predict(X_meta_test))

print(f"Logistic ({model_label}):", round(log_score, 4))
print(f"Random Forest ({model_label}):", round(rf_score, 4))
print("Stacked model:", round(stack_score, 4))


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalt

Logistic (Step 3 tuned): 0.7809
Random Forest (Step 3 tuned): 0.8006
Stacked model: 0.8002


Exception ignored in: <function ResourceTracker.__del__ at 0x1048e1bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x103a75bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x106f75bc0>
Traceback (most recent call last

**#5.** In a markdown cell at the end, evaluate your results by describing how performance changed as you reduced features, which features were consistently important, which features you removed and why, whether stacking improved performance, and what you learned about how your model behaves.

- I reduced the feature set from 377 to 20 (about 94.7% fewer features) and still kept strong performance, with best balanced accuracy reaching 0.8006

- As I reduced features, performance stayed competitive, which suggests the biggest signals were captured by a small core set of features rather than the full high-dimensional space

- Logistic Regression was very stable and changed only slightly with tuning (0.7805 baseline, 0.7805 tuned, 0.7809 Step 3 tuned)

- Random Forest responded much more to tuning (0.7759 baseline, 0.7998 tuned, 0.8006 Step 3 tuned), which showed tuning had a clear effect

- The most consistently important features were marital-status and also the interaction terms with age, occupation, hours-per-week, educational-num, relationship, race, and native-country

- I removed features that did not make the top 20 importance ranking because they added complexity and runtime without a clear gain in balanced accuracy

- Stacking gave a balanced accuracy score of 0.8002, which was slightly below the best base model (Random Forest at 0.8006), so stacking did not improve final performance for me in this run

- Overall, my models suggest this problem has meaningful nonlinear interactions which tree-based models captured better, while logistic regression was more stable but less sensitive to tuning